In [1]:
import pandas as pd
import os
import unicodedata
from datetime import datetime, timedelta
from difflib import get_close_matches

# === Paths ===
today = datetime.now().date()
yesterday = today - timedelta(days=1)

# File paths
game_scores_path = f"../data/game-scores/game_scores_{yesterday}.csv"
training_set_path = f"../training-data/training-set/training_set_{yesterday}.csv"

# === Team Abbreviation to Full Name Map ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KAN": "Kansas City Royals",
    "ARI": "Arizona Diamondbacks", "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres",
    "MIN": "Minnesota Twins", "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# === Normalize Function ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Fuzzy Matching Helper ===
def fuzzy_match(target, choices, cutoff=0.7):
    matches = get_close_matches(target, choices, n=1, cutoff=cutoff)
    return matches[0] if matches else None

# === Load Data ===
training_df = pd.read_csv(training_set_path)
scores_df = pd.read_csv(game_scores_path)

# === Create full name columns from team abbreviations ===
training_df['home_team_full'] = training_df['home_team'].map(team_name_map)
training_df['away_team_full'] = training_df['away_team'].map(team_name_map)

# === Normalize team names ===
training_df['home_team_clean'] = training_df['home_team_full'].apply(normalize_str)
training_df['away_team_clean'] = training_df['away_team_full'].apply(normalize_str)
scores_df['home_team_clean'] = scores_df['Home Team'].apply(normalize_str)
scores_df['away_team_clean'] = scores_df['Away Team'].apply(normalize_str)

# === Create index for fuzzy matching ===
scores_lookup = {(normalize_str(row['Home Team']), normalize_str(row['Away Team'])): row 
                 for _, row in scores_df.iterrows()}

# === Assign Scores ===
home_scores = []
away_scores = []
unmatched_games = []

for _, row in training_df.iterrows():
    key = (row['home_team_clean'], row['away_team_clean'])
    score_row = scores_lookup.get(key)

    # Try fuzzy match if exact key not found
    if score_row is None:
        matched_home = fuzzy_match(row['home_team_clean'], scores_df['home_team_clean'].tolist())
        matched_away = fuzzy_match(row['away_team_clean'], scores_df['away_team_clean'].tolist())

        if matched_home and matched_away:
            match = scores_df[(scores_df['home_team_clean'] == matched_home) &
                              (scores_df['away_team_clean'] == matched_away)]
            if not match.empty:
                score_row = match.iloc[0]

    # Try reversed teams if still not matched (to account for potential mislabels)
    if score_row is None:
        key_rev = (row['away_team_clean'], row['home_team_clean'])
        score_row = scores_lookup.get(key_rev)

    if score_row is not None:
        home_scores.append(score_row['Home Score'])
        away_scores.append(score_row['Away Score'])
    else:
        home_scores.append(None)
        away_scores.append(None)
        unmatched_games.append((row['home_team'], row['away_team']))

training_df['home_score'] = home_scores
training_df['away_score'] = away_scores
training_df['fav_score'] = training_df.apply(
    lambda row: row['home_score'] if row['home_favorite'] == 1 else row['away_score'], axis=1)
training_df['dog_score'] = training_df.apply(
    lambda row: row['away_score'] if row['home_favorite'] == 1 else row['home_score'], axis=1)

# === Drop helper columns ===
training_df.drop(columns=['home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean'], inplace=True)

# === Save Updated CSV ===
training_df.to_csv(training_set_path, index=False)
print(f"✅ Updated training set with game scores saved to {training_set_path}")

if unmatched_games:
    print("⚠️ Unmatched games:")
    for home, away in unmatched_games:
        print(f"- {home} vs {away}")


✅ Updated training set with game scores saved to ../training-data/training-set/training_set_2025-06-02.csv
